> **구분: 수업 후 복습·자습**

# 01. 얼굴·눈 landmark와 기초 시선 — 15분

**예상 시간: 15분**

이 노트북은 영상 처리를 처음 배우는 비전공 학부생을 기준으로 합니다. 설명을 외우기보다 **입력 → 배열 → 변환 → 저장 → 재검증** 순서를 따라가세요.

목표는 모델이 찾은 좌표와 우리가 만든 방향 규칙을 구분하고, 미검출·퇴화한 눈 폭을 안전한 상태로 표현하는 것입니다.

필요한 실행·저장 규칙은 이 노트북에서 다시 제공합니다. [07.Mediapipe_v2 README](README.md)와 [용어 찾아보기](GLOSSARY.md)는 추가 참고 자료입니다. TODO는 JupyterLab 파일 편집기에서 저장하고, 미완성 검증이 실패하는 것은 정상입니다. 참고 정답 셀은 강사가 안내하는 시점에 펼칩니다.

본수업에서는 환경 5분과 얼굴 시연 5분을 진행합니다. 아래 얼굴 구현 15분은 자습입니다. 상세 보정 내용은 자습으로 둡니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

얼굴 특징점은 얼굴의 위치에 대한 모델 출력이고, 그 점으로 시선 방향을 추정하는 규칙은 별도 프로그램입니다. 본수업은 짧은 관찰 시연이며 전체 구현은 자습입니다. 화면에서 고개·눈·거리·가림을 바꾸며 규칙의 한계를 확인합니다.

### 전체 과정에서 내 코드가 하는 일

```text
얼굴 영상
  ↓
특징점
  ↓
눈 안의 상대 위치 비율
  ↓
방향 규칙 TODO
  ↓
합성 경계 검사
  ↓
실제 조건 관찰
```

자습에서는 비율과 경계 규칙을 구현합니다. 이것을 보정된 정밀 시선 추적 장치로 해석하지 않고 미검출·mirror·특징점 불확실성을 함께 설명합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** LEFT가 보이면 실제로 왼쪽 물체를 바라본다는 뜻인가요?

**설명:** 이 교육용 규칙의 좌표 해석 결과입니다. 머리 방향·카메라 시점·mirror와 개인차가 영향을 줍니다. 좌우 기준과 실제 관찰 조건을 기록해야 하며 label 하나로 관심 대상까지 확정하지 않습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## A. 강사 설명 — 아직 코드를 실행하지 않습니다

**설명 자료:** [실습 전 이론·실행 안내 38쪽](../슬라이드/강의자료_3일차_v2.pdf#page=38) → [실습 시작 39쪽](../슬라이드/강의자료_3일차_v2.pdf#page=39) → ipynb 수행 → [실습 후 결과 해설 40쪽](../슬라이드/강의자료_3일차_v2.pdf#page=40).

강사는 아래 순서로 이론과 노트북 사용 방법을 설명합니다. 수강생은 입력과 예상 결과를 말로 예측합니다.

1. 이 실습은 무엇을 위한 것인가요?
2. 전체 과정에서 내 코드가 하는 일
3. 실행 전에 생각하고 결과로 설명하기
4. 앞 실습의 위치와 이번에 다시 필요한 내용
5. 본수업 강사 시연 — USB 카메라 얼굴·눈 관찰 (5분)
6. 1. 모델 출력과 규칙 출력을 분리합니다
7. 좌우 이름을 먼저 정의하세요
8. 2. 실제 양성 정적 입력

노트북에서 `얼굴·눈 landmark와 시선 추정의 한계`의 입력·처리·출력을 보여 주고, 실행 위치와 수정할 항목을 짚습니다. 입력 형식·API·실행 주의사항을 설명한 뒤, 아래 **B. 수강생 실습 시작**에서 실행을 시작하도록 안내합니다.

**완료 기준:** 노트북의 TODO·확인 질문과 결과 기록 완료

**선택 자습:** 본수업 필수 완료 범위 밖입니다. 선택 수업으로 운영할 때에도 설명 → 실습 → 결과 해설·확인 순서를 따릅니다.


## B. 수강생 실습 시작

강사의 설명과 실습 방법 시연이 끝나면 여기부터 진행합니다. 아래 이론 문단은 실습 중 참고할 설명입니다.

1. 이 노트북의 커널을 새로 시작하고 첫 준비·import 셀부터 실행합니다. 다른 단원의 `common` 모듈이 남아 있지 않도록 노트북별 새 커널을 사용합니다.
2. **TODO 전 따라 실행하기**의 완성 예제를 수정 없이 한 셀씩 실행하고 출력·관찰 설명을 확인합니다. 그런 다음 TODO에서 기능을 조합합니다. 터미널 명령은 표시된 호스트/컨테이너 터미널에 입력합니다.
3. 다음 v2 폴더의 파일을 수정·저장한 뒤 노트북의 학습자 검증 셀을 실행합니다: [exercises/gaze.py](exercises/gaze.py).
4. 실제 입력 실행·결과 저장을 마친 뒤 C에서 [결과 해설 40쪽](../슬라이드/강의자료_3일차_v2.pdf#page=40)으로 돌아가 출력·오류를 비교하고 완료 기록을 작성합니다. 참고 정답은 자신의 시도 후 강사 안내에 따라 비교합니다.

**실행 범위:** 얼굴·눈 landmark와 시선 추정의 한계


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| 모델/이미지 파일 없음·입력을 열지 못함 | 초기 probe 결과와 오류의 경로를 확인합니다. README 자산과 제공 이미지부터 읽습니다. | 제공 이미지 한 장 처리 성공 |
| 미검출·시선 unknown | 조명·얼굴 크기·가림과 눈 좌표의 유효성을 봅니다. 미검출은 반드시 실행 오류는 아닙니다. | 합성 눈 fixture 검사와 실제 결과를 구분 |
| 표시 환경 없음·CalledProcessError | 원격은 예시의 --headless를 유지하고 subprocess stderr를 읽습니다. NORM_RECT 경고만으로 실패라고 판단하지 않습니다. | 종료 코드 0과 실제 결과 JSON 확인 |

**예시에 없는 오류를 확인할 곳:** [MediaPipe README](README.md), [run_tasks.py](scripts/run_tasks.py), `Data/logs/07.Mediapipe_v2/face_results.json`. [공식 Face Landmarker 안내](https://developers.google.com/edge/mediapipe/solutions/vision/face_landmarker/python).


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**MediaPipe 준비:** Ollama·ROS node는 필요 없습니다. 배포 자산은 `Data/models/07.Mediapipe/{face_landmarker.task,hand_landmarker.task}`, `Data/images/07.Mediapipe/person_open_palm.png`, `Data/videos/07.Mediapipe/person_open_palm.avi`입니다. 첫 Python 셀의 환경 검사가 모델·입력 해시와 실제 검출을 확인합니다. 파일이 없거나 검사에 실패하면 해당 배포 경로·수업 이미지를 확인한 뒤 재실행합니다. 입력은 사전 제공 파일이며 다른 노트북의 실행 결과를 요구하지 않습니다.

얼굴·손 카메라 시연 전에도 **컨테이너 JupyterLab Terminal** 에서 `cd /workspace` 후 `/opt/course/bin/python 07.Mediapipe_v2/scripts/check_environment.py`로 준비를 확인할 수 있습니다. USB·GUI는 로컬 한 곳에서 실행하고 두 수강생이 함께 봅니다. 결과 영상은 `Data/outputs/07.Mediapipe_v2`, 상태 JSON은 `Data/logs/07.Mediapipe_v2`에 남습니다. 같은 이름의 재실행 결과가 갱신되므로 비교할 파일은 먼저 별도 이름으로 보관합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[06.OpenCV_v2/02_colors_drawing.ipynb의 1절](../06.OpenCV_v2/02_colors_drawing.ipynb)에 BGR·RGB 구분이 있습니다. 필요한 내용은 OpenCV가 BGR, MediaPipe 입력이 RGB이며 제공 `TasksDetector`가 변환을 수행한다는 것입니다. landmark는 검출된 점의 좌표이고 normalized x·y는 영상 너비·높이에 대한 비율입니다. 이번 TODO는 이미 주어진 눈 양 끝·홍채 x,y를 받아 방향 규칙만 구현하므로 OpenCV 결과 파일이나 별도 모델 학습은 필요 없습니다.

<!-- notebook-restart:end -->

## 본수업 강사 시연 — USB 카메라 얼굴·눈 관찰 (5분)

얼굴 TODO를 구현하기 전에 제공된 실행 프로그램으로 시연합니다. **로컬 사용자의 컨테이너 JupyterLab Terminal** 에서 아래 명령을 실행합니다. USB 카메라는 Jetson에 연결되어 있어야 하며, 앞의 OpenCV·손 카메라 프로그램은 먼저 종료합니다. 두 수강생이 Jetson 로컬 모니터를 함께 봅니다.

```bash
cd /workspace/07.Mediapipe_v2
/opt/course/bin/python scripts/run_tasks.py \
  --mode face --source camera --camera-index 0 \
  --mirror --max-frames 10000
```

**진행 순서:** ① 정면 얼굴을 비추고 눈 양 끝·홍채 중심에 그려지는 점과 `LOOK_CENTER` 등의 상태를 봅니다. ② 머리를 가급적 고정하고 눈동자를 화면 좌우로 움직여 `LOOK_LEFT`·`LOOK_RIGHT` 변화를 관찰합니다. ③ 얼굴을 화면 밖으로 옮겨 `NO_FACE`를 확인합니다. ④ 머리 회전·안경·조명에 따라 판정이 달라질 수 있음을 설명합니다. 기대 상태가 나오지 않으면 실제 결과와 조건을 말하며 성공으로 간주하지 않습니다.

**강의 멘트:** “모델은 특징점 좌표를 찾고, 규칙은 눈 양 끝 사이의 홍채 위치 비율로 상태를 정합니다. LEFT/RIGHT는 검출에 넣은 화면 기준이며 사람의 실제 시선 방향을 정밀하게 측정한 결과가 아닙니다.” `--mirror`는 검출 전에 영상을 좌우 반전합니다. 실시간 창에는 점과 label이 표시되며, 수치 `ratio`는 저장 JSON에서 확인합니다.

시연은 약 2분 관찰 후 **영상 창에 초점을 두고 `q` 또는 Esc** 로 종료하고 나머지 시간에 결과와 한계를 설명합니다. `--max-frames 10000`은 자동 종료 상한이지 5분 타이머가 아닙니다. 종료 후 영상은 `Data/outputs/07.Mediapipe_v2/face_video.avi`, 상태·ratio 기록은 `Data/logs/07.Mediapipe_v2/face_results.json`에 저장됩니다. 같은 명령을 다시 실행하면 같은 이름의 결과가 갱신되므로 비교할 결과는 먼저 별도 이름으로 복사합니다.

창이 없으면 원격 컨테이너가 아닌 로컬 컨테이너와 Jetson 모니터를 사용하는지 확인합니다. 카메라 열기 실패 시 다른 프로그램 점유와 장치 번호를 확인하고 `--camera-index`를 실제 번호로 바꿉니다. 얼굴이 검출되지 않으면 얼굴 크기·조명·가림을 먼저 확인합니다.

이 시연은 제공된 `common/geometry.py`의 판정을 사용하므로 얼굴 TODO가 미완성이어도 실행됩니다. 종료 후 [손 제스처 실습](02_hand_gestures.ipynb)으로 이동합니다. 아래 얼굴 구현은 자습입니다.

**슬라이드 위치:** [얼굴 시연 38쪽](../슬라이드/강의자료_3일차_v2.pdf#page=38) → 시연 실행 → [관찰 결과 해설 40쪽](../슬라이드/강의자료_3일차_v2.pdf#page=40). 본수업은 시연·해설을 합쳐 5분이며 손 노트북으로 이어집니다. 얼굴 TODO는 [선택 자습 시작 39쪽](../슬라이드/강의자료_3일차_v2.pdf#page=39)을 참고합니다.


## 정답 활용 안내

> 교재 버전: v2 (실행 이미지 v1.0)

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

본수업은 손 제스처 중심입니다. 얼굴·시선은 강사가 5분 동안 결과와 한계를 시연하고, 이 노트북의 15분 구현 경로는 자습으로 진행합니다. 얼굴 TODO 완료는 07 본수업 통과 조건이 아닙니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 손 상태·미검출 횟수와 결과 영상을 저장합니다.
- **단원 마지막 연결 질문:** 한 장면에서 성공한 것과 여러 사람·가림·조명에서 쓸 수 있는 것을 구분합니다. 손 제스처는 이 사례의 선택 입력이며 부품 검사 모델은 아닙니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import json, subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '07.Mediapipe_v2/common').is_dir())
COURSE = ROOT / '07.Mediapipe_v2'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.contracts import check
from common.geometry import estimate_gaze, classify_hand
from common.synthetic import make_eye, make_hand, draw_hand
from common.runtime import OUTPUT_DIR
probe = subprocess.run([sys.executable, str(COURSE/'scripts/check_environment.py')], text=True, capture_output=True, timeout=60)
print(probe.stdout)
if probe.stderr: print('MediaPipe runtime log:\n'+probe.stderr[-1500:])
probe.check_returncode()


> **구분: 수업 후 복습·자습**

## 1. 모델 출력과 규칙 출력을 분리합니다

<!-- procedure:lesson-24-059966:start -->
![얼굴 특징점에서 규칙 해석까지](../교재그림/절차/lesson-24.svg)

시선 방향의 보조 규칙이며 실제 주시 대상이나 의도를 알아냈다고 해석하지 않습니다.
<!-- procedure:lesson-24-059966:end -->


Face Landmarker는 이미지에서 얼굴을 찾고 여러 normalized landmark를 냅니다. `runtime.py`는 한쪽 눈 끝 33·133과 iris 468–472의 평균을 고릅니다. 이 번호는 모델 계약이고, 아래 ratio와 label은 우리 수업 코드의 계약입니다.

모델이 얼굴을 놓친 경우 좌표 계산을 억지로 하지 않고 `NO_FACE`를 반환합니다. 눈 양 끝 x가 같아 분모가 0이면 `UNKNOWN`입니다. 정상 상태만 `status=OK`입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
for ratio in (0.25, 0.42, 0.50, 0.58, 0.75):
    frame, left, right, iris = make_eye(ratio)
    print('input ratio=', ratio, 'result=', estimate_gaze(left, right, iris))
print('missing=', estimate_gaze(None, None, None))


> **구분: 수업 후 복습·자습**

### 좌우 이름을 먼저 정의하세요

이 자료의 LEFT/RIGHT는 **검출에 넣은 이미지 화면 방향** 입니다. `--mirror`로 먼저 뒤집으면 뒤집힌 화면이 기준입니다. 보는 사람의 왼눈/오른눈, 사용자가 실제로 바라본 공간 방향과 자동으로 같지 않습니다.


> **구분: 수업 후 복습·자습**

## 2. 실제 양성 정적 입력

`person_open_palm.png`는 실제 수강생 사진이 아니라 수업 검증용으로 AI 생성한 photorealistic 이미지입니다. 파일 출처 설명과 SHA-256은 `assets-manifest.json`에 고정되어 있습니다. 같은 입력에서 Face Tasks와 좌표 규칙을 함께 실행해 `LOOK_CENTER`가 나오는지 확인합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
positive = ROOT/'Data/images/07.Mediapipe/person_open_palm.png'
command = [sys.executable, str(COURSE/'scripts/run_tasks.py'),
           '--mode', 'face', '--source', str(positive), '--headless']
run = subprocess.run(command, text=True, capture_output=True, timeout=90)
print(run.stdout)
if run.stderr: print(run.stderr[-1500:])
run.check_returncode()
report = json.loads((ROOT/'Data/logs/07.Mediapipe_v2/face_results.json').read_text())
assert report['frames'] == 1 and report['counts'] == {'LOOK_CENTER': 1}


## TODO 전 따라 실행하기 — 코드 채우기 없음

> **구분: 수업 후 복습·자습**

**완성 예제 실행 → 출력 관찰 → TODO에서 조합** 순서입니다. 아래 예제를 위에서부터 그대로 실행하고 관찰 질문에 답하세요. 예제의 `warm_` 변수는 연습용입니다. 기존 실습 시간 안에서 먼저 진행하며, TODO 구현·검증 시간과 마지막 결과 정리 시간을 확보합니다.


### 따라 실행 1. 좌표 인덱스·정렬·비율 계산

좌표 튜플의 [0]은 x입니다. 눈 양 끝의 입력 순서가 바뀌어도 작은 x부터 계산합니다. 실제 얼굴 검출과 분리한 숫자 예제입니다.


In [ ]:
# 완성 예제: 수정 없이 실행합니다.
import math
warm_corners = [(200.0, 50.0), (100.0, 50.0)]
warm_iris = (140.0, 50.0)
warm_min, warm_max = sorted(point[0] for point in warm_corners)
warm_span = warm_max - warm_min
print('작은 x / 큰 x / 너비:', warm_min, warm_max, warm_span)
print('유한 좌표:', all(math.isfinite(v) for point in warm_corners for v in point))
warm_ratio = (warm_iris[0] - warm_min) / warm_span
print('홍채 비율:', warm_ratio)
print('None 검출:', any(point is None for point in [warm_corners[0], None, warm_iris]))
print('퇴화 너비:', abs(100.0 - 100.0) <= 1e-9)


**실행 후 관찰:** 비율은 0.4입니다. None과 너비 0을 먼저 검사해야 좌표 접근·나눗셈 오류를 피할 수 있습니다.


### 따라 실행 2. 경계 비교·round: 비율에 이름 붙이기

경계와 같은 값은 중앙에 포함합니다. 반올림은 분류한 뒤 반환할 숫자에 적용합니다.


In [ ]:
# 완성 예제: 수정 없이 실행합니다.
for warm_value in (0.41, 0.42, 0.50, 0.58, 0.59):
    if warm_value < 0.42:
        warm_label = 'LOOK_LEFT'
    elif warm_value > 0.58:
        warm_label = 'LOOK_RIGHT'
    else:
        warm_label = 'LOOK_CENTER'
    print(warm_value, '→', warm_label, '반환 숫자:', round(warm_value, 6))


**실행 후 관찰:** 0.42와 0.58은 LOOK_CENTER입니다. TODO에서는 누락·잘못된 좌표·퇴화 너비를 검사한 뒤 ratio와 status, label을 반환합니다.


### 이제 TODO에서 조합합니다

방금 실행한 기능을 아래 TODO의 입력·출력 계약에 맞게 연결하세요. 고정된 예제 값 대신 함수 인자 또는 지정된 실습 입력을 사용합니다. 필요한 import와 오류 처리도 직접 구성합니다. 아래 학습자 검증을 통과하고 자신의 결과를 저장한 뒤 실습 후 PDF 해설로 돌아갑니다.


> **구분: 수업 후 복습·자습**

## 자습 구현 TODO — [gaze.py](exercises/gaze.py)

수정 파일: [exercises/gaze.py](exercises/gaze.py)

작은 corner x와 큰 corner x 사이의 iris 비율을 계산합니다. 정확히 0.42와 0.58은 CENTER이며, 좌표 순서를 뒤집어 입력해도 같은 결과여야 합니다.

JupyterLab 파일 편집기에서 TODO를 구현하고 **저장** 한 뒤 아래 검증을 실행합니다. 검증은 저장된 모듈을 새 이름으로 다시 읽으므로 일반적으로 kernel 재시작은 필요 없습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `estimate_gaze(left_corner,right_corner,iris_center,*,center_low=.42,center_high=.58)`는 각 점의 x,y를 읽고 `{'status':..., 'label':..., 'ratio':...}`를 반환합니다. 정상 label은 정확히 `LOOK_LEFT/LOOK_CENTER/LOOK_RIGHT`입니다.
**작성 흐름 힌트:** 하나라도 None이면 NO_FACE·ratio=None → 유효 임계값/유한 좌표 검사 → 두 corner x 정렬 → 너비가 1e-9 이하면 UNKNOWN·ratio=None → `(iris_x-x_min)/width` → 경계로 분류하고 ratio를 소수 6자리로 반환. 정상 status는 OK입니다.
**손으로 확인:** 양 끝 x=0,1이고 홍채 x=.42이면 `{'status':'OK','label':'LOOK_CENTER','ratio':0.42}`입니다. .58도 중앙에 포함합니다.
**설명할 지점:** “좌우는 사람의 해부학적 방향이 아니라 입력 화면의 방향입니다. 손 제스처 본수업과 구분한 자습 항목입니다.”
**검사와 분류를 빠짐없이 구현하기:** 각 좌표는 최소 x,y 두 값이며 `float` 변환 후 둘 다 유한해야 합니다. `math.isfinite`를 쓰려면 직접 `import math`합니다. 임계값은 `0<center_low<center_high<1`, 위반은 `ValueError`입니다. None은 `{'status':'NO_FACE','label':'NO_FACE','ratio':None}`, 눈 폭≤1e-9는 `{'status':'UNKNOWN','label':'UNKNOWN','ratio':None}`입니다. 정상은 반올림 **전** ratio<low이면 LOOK_LEFT, ratio>high이면 LOOK_RIGHT, 그 외 LOOK_CENTER로 정하고 반환 ratio만 `round(ratio,6)`합니다. 범위 밖 ratio를 임의로 0~1에 자르지 않습니다. `check('gaze','exercise')`에서 미검출·좌우 입력 역순·0.42/0.58 경계·퇴화 폭·잘못된 임계값을 확인합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE/'exercises/gaze.py').read_text())


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('gaze', 'exercise')
print('learner_verification=PASS')


> **구분: 수업 후 복습·자습**

### 참고 정답

강사가 안내하는 시점에 정답과 해설을 비교합니다. 정답은 별도 파일이며 exercise를 덮어쓰지 않습니다. 구현이 한 글자씩 같을 필요는 없고 입력·출력·오류 처리 계약이 같으면 됩니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE/'solutions/gaze.py').read_text())


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/gaze.py](solutions/gaze.py)의 `estimate_gaze`에서 입력 검사와 계산을 직접 읽습니다.

**수평 비율:** 눈 양 끝 x를 작은 값·큰 값으로 정리하고 `(iris_x-x_min)/(x_max-x_min)`을 계산합니다. x=100·200 사이 홍채가 150이면 ratio=0.5여서 LOOK_CENTER입니다. 0.42 미만은 LEFT, 0.58 초과는 RIGHT이고 경계 두 값은 CENTER에 포함됩니다.

좌표가 없으면 NO_FACE, 눈 폭이 거의 0이면 UNKNOWN입니다. 표시 방향은 입력 영상 기준이므로 미러 영상에서는 사람이 생각하는 좌우와 다를 수 있습니다. 얼굴 자세·개인차 보정이 없는 기초 규칙이며 정밀 시선 추적 정확도를 뜻하지 않습니다.

**확인 질문:** 눈 폭으로 나누는 이유와, 고개를 돌렸을 때 같은 비율을 그대로 해석하기 어려운 이유를 설명하세요. 얼굴 구현은 자습입니다.

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/gaze.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 눈 양 끝 → 홍채 위치 비율 → 화면 기준 시선 상태.
None은 NO_FACE, 눈 너비가 거의 0이면 UNKNOWN, 임계값 경계는 CENTER입니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
import math
from typing import Sequence

Point = tuple[float, float]


NO_FACE = {"status": "NO_FACE", "label": "NO_FACE", "ratio": None}


# 좌표 두 값을 실수로 읽고 NaN·Inf를 거부합니다.
# 잘못된 좌표가 뒤의 거리·비율 계산까지 퍼지지 않도록 입구에서 확인합니다.
def _point(value: Sequence[float], name: str) -> Point:
    if len(value) < 2:
        raise ValueError(f"{name}에는 x, y가 필요합니다.")
    x, y = float(value[0]), float(value[1])
    if not (math.isfinite(x) and math.isfinite(y)):
        raise ValueError(f"{name} 좌표는 유한한 숫자여야 합니다.")
    return x, y


def estimate_gaze(
    left_corner: Sequence[float] | None,
    right_corner: Sequence[float] | None,
    iris_center: Sequence[float] | None,
    *,
    center_low: float = 0.42,
    center_high: float = 0.58,
) -> dict[str, str | float | None]:
    """눈 양 끝 사이에서 홍채 중심의 수평 비율을 분류한다.

    LEFT/RIGHT는 **입력 이미지의 화면 방향**이다. 셀피 미러 표시나 사람의
    해부학적 좌우와 자동으로 같다고 가정하지 않는다.
    """
    # 미검출은 좌표 계산 전에 반환합니다. 검출 실패와 유효 좌표의 중앙 시선을
    # 같은 상태로 처리하면 결과를 잘못 해석할 수 있기 때문입니다.
    if left_corner is None or right_corner is None or iris_center is None:
        return dict(NO_FACE)
    if not (0.0 < center_low < center_high < 1.0):
        raise ValueError("0 < center_low < center_high < 1 조건이 필요합니다.")
    first = _point(left_corner, "left_corner")
    second = _point(right_corner, "right_corner")
    iris = _point(iris_center, "iris_center")
    # 인자 순서와 무관하게 화면의 작은 x부터 비율을 측정합니다.
    x_min, x_max = sorted((first[0], second[0]))
    width = x_max - x_min
    # 눈 너비가 거의 0이면 비율을 계산할 수 없습니다.
    # 0으로 나누거나 불안정한 큰 값으로 분류하지 않고 UNKNOWN을 반환합니다.
    if width <= 1e-9:
        return {"status": "UNKNOWN", "label": "UNKNOWN", "ratio": None}
    ratio = (iris[0] - x_min) / width
    # 눈 너비로 정규화한 비율을 경계와 비교합니다.
    # 등호는 중앙에 포함하고 반올림은 반환할 숫자에만 적용해 경계 판정을 보존합니다.
    if ratio < center_low:
        label = "LOOK_LEFT"
    elif ratio > center_high:
        label = "LOOK_RIGHT"
    else:
        label = "LOOK_CENTER"
    return {"status": "OK", "label": label, "ratio": round(float(ratio), 6)}
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('gaze', 'solution')
print('solution_verification=PASS')


> **구분: 수업 후 복습·자습**

## 결과를 과장하지 않기

이 ratio는 head pose와 개인별 calibration을 쓰지 않습니다. 따라서 `LOOK_LEFT`는 화면 안 홍채 위치에 대한 기초 분류이며 정밀 시선 추적 결과가 아닙니다. 한 사람의 한 장면 성공으로 정확도를 말할 수 없고, 안경·조명·머리 회전·거리 조건을 나눠 데이터를 모아야 합니다.


> **구분: 수업 후 복습·자습**

## 자습 — 두 눈과 시간 안정화

두 눈의 ratio가 모두 유효할 때 평균하고, 서로 크게 다르면 UNKNOWN으로 두는 규칙을 설계할 수 있습니다. 최근 5 frame 다수결은 흔들림을 줄이지만 반응을 늦춥니다. 어떤 상태에서도 오래된 방향을 무한 유지하지 않도록 결과 timestamp와 유효 시간을 후속 ROS 메시지에 포함하는 방법을 생각해 보세요.


> **구분: 수업 후 복습·자습**

## 완료 질문

1. ratio 0.5는 confidence 50%가 아닌 이유는 무엇인가요?
2. 얼굴 미검출을 CENTER로 바꾸면 후속 제어에서 어떤 문제가 생길까요?
3. `--mirror`를 검출 전 적용했는지 후 적용했는지가 왜 중요할까요?


## C. 실습 후 결과 해설·완료 확인

**슬라이드 복귀:** [실습 후 결과 해설 40쪽](../슬라이드/강의자료_3일차_v2.pdf#page=40).

실행한 셀의 출력·저장 결과를 열어 두고 강사 해설과 비교합니다. 예상과 달랐던 점·미완료 항목을 기록한 뒤 다음 이론으로 이동합니다.

노트북의 TODO·확인 질문과 결과 기록 완료

아래 항목을 본인의 결과로 채우고 노트북을 저장합니다.

| 항목 | 내 기록 |
|---|---|
| 실행한 입력·조건 | 작성 |
| 수정한 코드 또는 답안 | 작성 |
| 결과 파일·검증 출력 | 작성 |
| 예상과 달랐던 결과·이유 | 작성 |
| 미완료 항목·오류·다음 조치 | 작성 |

강사와 완료 기준을 확인한 후 다음 단원의 설명으로 이동합니다.
